In [7]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import implicit
import itertools
import random

train_df = pd.read_csv("data/split/train_split.csv")
val_df   = pd.read_csv("data/split/val_split.csv")
test_df  = pd.read_csv("data/split/test_split.csv")

steam = pd.read_csv("steam_store/steam.csv")
steam["genres"] = steam["genres"].fillna("").apply(lambda x: x.split(";"))
steam_small = steam[["appid", "genres"]].copy()

print("Train:", train_df.shape, "users:", train_df["user_id"].nunique())
print("Val:  ", val_df.shape,   "users:", val_df["user_id"].nunique())
print("Test: ", test_df.shape,  "users:", test_df["user_id"].nunique())


Train: (63905, 8) users: 9906
Val:   (11649, 8) users: 9906
Test:  (12784, 8) users: 9906


In [ ]:
def to_bool(x):
    if isinstance(x, str):
        return x.lower() == "true"
    return bool(x)

for df in (train_df, val_df, test_df):
    if "is_recommended" in df.columns:
        df["is_recommended"] = df["is_recommended"].apply(to_bool)
    else:
        df["is_recommended"] = False


def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    return sum((i in rel_set) for i in rec_k) / len(rec_k)


def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    return sum((i in rel_set) for i in rec_k) / len(rel_set)


def ndcg_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return dcg / idcg if idcg > 0 else 0.0


def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0


def map_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    ap_sum, hits = 0.0, 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0


def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

In [ ]:

train_base = train_df.copy()
train_base["hours"] = train_base["hours"].clip(lower=0.0)

# users
user_cats = train_base["user_id"].astype("category")
train_base["user_idx"] = user_cats.cat.codes
idx2user = dict(enumerate(user_cats.cat.categories))
user2idx = {u: i for i, u in idx2user.items()}

# items
item_cats = train_base["app_id"].astype("category")
train_base["item_idx"] = item_cats.cat.codes
idx2item = dict(enumerate(item_cats.cat.categories))
item2idx = {i: j for j, i in idx2item.items()}

n_users = len(idx2user)
n_items = len(idx2item)

print(f"\nusuarios distintos en train: {n_users}")
print(f"items distintos en train:   {n_items}")

# extraer géneros
train_base["genres"] = train_base["app_id"].map(
    steam_small.set_index("appid")["genres"]
).fillna("").apply(lambda x: x if isinstance(x, list) else [])

train_base["genre_weight"] = train_base["genres"].apply(len).astype(float)

row_idx = train_base["item_idx"].values
col_idx = train_base["user_idx"].values


usuarios distintos en train: 9906
items distintos en train:   2822


In [ ]:
def build_rel_idx(df, user2idx, item2idx):
    df2 = df.copy()
    # solo usuarios/items que conocemos
    df2 = df2[df2["user_id"].isin(user2idx)]
    df2 = df2[df2["app_id"].isin(item2idx)]

    df2["user_idx"] = df2["user_id"].map(user2idx)
    df2["item_idx"] = df2["app_id"].map(item2idx)

    # todos los usuarios presentes en el split
    all_user_idx = df2["user_idx"].unique()

    # relevancia
    df_rel = df2[df2["is_recommended"] == True]

    rel_true = (
        df_rel
        .groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    # devolvemos un dict con todos los usuarios, set() si no tiene ningún True
    rel = {u_idx: rel_true.get(u_idx, set()) for u_idx in all_user_idx}
    return rel

val_rel_idx  = build_rel_idx(val_df,  user2idx, item2idx)
test_rel_idx = build_rel_idx(test_df, user2idx, item2idx)

print("\nusuarios en val_rel_idx: ", len(val_rel_idx))
print("usuarios en test_rel_idx:", len(test_rel_idx))

# items vistos en train
user_seen = train_base.groupby("user_idx")["item_idx"].apply(set).to_dict()

K = 10


usuarios en val_rel_idx:  9823
usuarios en test_rel_idx: 9664


In [11]:
def run_bpr(alpha, beta, gamma, factors, lr, reg, iterations):
    # pesos de confianza
    conf_base    = 1.0 + alpha * np.log1p(train_base["hours"].values)
    rec_factor   = 1.0 + beta  * train_base["is_recommended"].astype(float).values
    genre_factor = 1.0 + gamma * train_base["genre_weight"].values

    confidence = (conf_base * rec_factor * genre_factor).astype(np.float32)

    # matriz item-user con pesos
    item_user_mat = csr_matrix(
        (confidence, (row_idx, col_idx)),
        shape=(n_items, n_users)
    )

    model = implicit.bpr.BayesianPersonalizedRanking(
        factors=factors,
        learning_rate=lr,
        regularization=reg,
        iterations=iterations
    )

    model.fit(item_user_mat)

    precs, recs, ndcgs = [], [], []
    n_items_model = model.item_factors.shape[0]

    for u_idx, rel_items in val_rel_idx.items():
        if u_idx >= model.user_factors.shape[0]:
            continue

        scores = model.user_factors[u_idx] @ model.item_factors.T
        seen = user_seen.get(u_idx, set())

        ranked = [(it, scores[it]) for it in range(n_items_model) if it not in seen]
        if not ranked:
            continue

        ranked.sort(key=lambda x: x[1], reverse=True)
        topk = [it for it, sc in ranked[:K]]

        precs.append(precision_at_k(topk, rel_items))
        recs.append(recall_at_k(topk, rel_items))
        ndcgs.append(ndcg_at_k(topk, rel_items))

    if not recs:
        return model, 0.0, 0.0, 0.0

    return model, float(np.mean(precs)), float(np.mean(recs)), float(np.mean(ndcgs))


In [12]:

alpha_grid   = [1.0, 2.0, 5.0]
beta_grid    = [0.5, 1.0, 2.0]
gamma_grid   = [0.0, 0.2, 0.5, 1.0]
factors_grid = [32, 64, 128, 256]
lr_grid      = [0.005, 0.01, 0.05]
reg_grid     = [0.001, 0.01, 0.05]
iter_grid    = [20, 50]

param_space = list(itertools.product(
    alpha_grid,
    beta_grid,
    gamma_grid,
    factors_grid,
    lr_grid,
    reg_grid,
    iter_grid
))

n_random_iter = 30
random_seed = 42

random.seed(random_seed)
sampled_configs = random.sample(param_space, n_random_iter)

best_model = None
best_cfg = None
best_recall = -1.0
results = []

print("\nIniciando RANDOM SEARCH BPR+GENRES (30 configs)...\n")

for (alpha, beta, gamma, f, lr, reg, iters) in sampled_configs:
    print(f"BPR alpha={alpha}, beta={beta}, gamma={gamma}, f={f}, lr={lr}, reg={reg}, it={iters}")

    model, prec, rec, ndcg = run_bpr(alpha, beta, gamma, f, lr, reg, iters)

    results.append({
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "factors": f,
        "lr": lr,
        "reg": reg,
        "iters": iters,
        "prec_val": prec,
        "recall_val": rec,
        "ndcg_val": ndcg
    })

    if rec > best_recall:
        best_recall = rec
        best_model = model
        best_cfg = (alpha, beta, gamma, f, lr, reg, iters)

    print(f" -> PREC@10_val={prec:.4f}, RECALL@10_val={rec:.4f}, NDCG@10_val={ndcg:.4f}\n")

results_df = pd.DataFrame(results).sort_values("recall_val", ascending=False)
results_df.head()


Iniciando RANDOM SEARCH BPR+GENRES (30 configs)...

BPR alpha=1.0, beta=1.0, gamma=0.5, f=64, lr=0.01, reg=0.001, it=20


100%|██████████| 20/20 [00:00<00:00, 119.40it/s, train_auc=50.44%, skipped=2.01%]


 -> PREC@10_val=0.0001, RECALL@10_val=0.0011, NDCG@10_val=0.0003

BPR alpha=1.0, beta=0.5, gamma=0.2, f=64, lr=0.05, reg=0.001, it=20


100%|██████████| 20/20 [00:00<00:00, 188.38it/s, train_auc=90.97%, skipped=1.91%]


 -> PREC@10_val=0.0000, RECALL@10_val=0.0001, NDCG@10_val=0.0000

BPR alpha=2.0, beta=0.5, gamma=1.0, f=128, lr=0.01, reg=0.05, it=20


...

100%|██████████| 50/50 [00:00<00:00, 394.94it/s, train_auc=50.45%, skipped=1.94%]


 -> PREC@10_val=0.0000, RECALL@10_val=0.0000, NDCG@10_val=0.0000

BPR alpha=5.0, beta=0.5, gamma=0.0, f=32, lr=0.005, reg=0.01, it=50


100%|██████████| 50/50 [00:00<00:00, 231.63it/s, train_auc=50.82%, skipped=2.03%]


 -> PREC@10_val=0.0000, RECALL@10_val=0.0000, NDCG@10_val=0.0000



,alpha,beta,gamma,factors,lr,reg,iters,prec_val,recall_val,ndcg_val
27,1.0,0.5,0.0,64,0.01,0.010,20,0.000250,0.002503,0.001078
17,5.0,2.0,0.5,32,0.05,0.050,50,0.000179,0.001788,0.000725
12,1.0,0.5,0.2,128,0.05,0.010,20,0.000179,0.001519,0.000657
22,2.0,2.0,1.0,256,0.01,0.010,20,0.000143,0.001430,0.000461
9,5.0,2.0,0.2,128,0.01,0.001,20,0.000179,0.001341,0.000479


In [13]:
print("\nmejor config en validacion:")
print(best_cfg)


mejor config en validacion:
(1.0, 0.5, 0.0, 64, 0.01, 0.01, 20)


In [18]:
def evaluate_model_on_split(model, rel_idx, user_seen, k=10):
    precs, recs, ndcgs, hits, maps = [], [], [], [], []
    rec_k_dict = {}

    total_users = len(rel_idx)
    skipped_out_of_range = 0
    skipped_no_ranked = 0

    n_items_model = model.item_factors.shape[0]

    for u_idx, rel_items in rel_idx.items():
        # evitar usuarios sin embedding
        if u_idx >= model.user_factors.shape[0]:
            skipped_out_of_range += 1
            continue

        scores = model.user_factors[u_idx] @ model.item_factors.T

        seen = user_seen.get(u_idx, set())
        ranked = [(it, scores[it]) for it in range(n_items_model) if it not in seen]

        if not ranked:
            skipped_no_ranked += 1
            continue

        ranked.sort(key=lambda x: x[1], reverse=True)
        topk = [it for it, sc in ranked[:k]]

        rec_k_dict[u_idx] = topk

        precs.append(precision_at_k(topk, rel_items))
        recs.append(recall_at_k(topk, rel_items))
        ndcgs.append(ndcg_at_k(topk, rel_items))
        hits.append(hit_score_at_k(topk, rel_items))
        maps.append(map_at_k(topk, rel_items))

    n_eval = len(rec_k_dict)

    print("---- evaluate_model_on_split (BPR) ----")
    print("usuarios en rel_idx:", total_users)
    print("saltados por user_factors (out rng):", skipped_out_of_range)
    print("saltados por ranked vacío:", skipped_no_ranked)
    print("suarios efectivamente evaluados:", n_eval)

    return (
        float(np.mean(precs)) if precs else np.nan,
        float(np.mean(recs))  if recs  else np.nan,
        float(np.mean(ndcgs)) if ndcgs else np.nan,
        float(np.mean(hits))  if hits  else np.nan,
        float(np.mean(maps))  if maps  else np.nan,
        rec_k_dict
    )

print("\nevaluando mejor modelo BPR en test")

prec_test, rec_test, ndcg_test, hit_test, map_test, rec_k_dict = \
    evaluate_model_on_split(best_model, test_rel_idx, user_seen, k=10)

info_videojuegos = {}
for idx, app in idx2item.items():
    genres = steam_small[steam_small["appid"] == app]["genres"]
    if len(genres) > 0:
        lista = genres.values[0]
        genero_principal = lista[0] if len(lista) > 0 else "desconocido"
    else:
        genero_principal = "desconocido"
    info_videojuegos[idx] = (app, genero_principal)

div_test = diversity_at_k(rec_k_dict, info_videojuegos)


evaluando mejor modelo BPR en test
---- evaluate_model_on_split (BPR) ----
usuarios en rel_idx: 9664
saltados por user_factors (out rng): 6913
saltados por ranked vacío: 0
suarios efectivamente evaluados: 2751


In [17]:

print(f"precision@10:{prec_test:.6f}")
print(f"recall@10:{rec_test:.6f}")
print(f"F1@10:{2 * (prec_test * rec_test) / (prec_test + rec_test + 1e-10):.6f}")
print(f"NDCG@10: {ndcg_test:.6f}")
print(f"Hit@10: {hit_test:.6f}")
print(f"MAP@10: {map_test:.6f}")
print(f"Diversity@10: {div_test:.6f}")


precision@10:0.000327
recall@10:0.002635
F1@10:0.000582
NDCG@10: 0.001192
Hit@10: 0.003272
MAP@10: 0.000659
Diversity@10: 1.000000
